In [ ]:
# CELL 1: Install Chrome + Dependencies
import sys
!apt-get update -qq
!apt-get install -yqq wget unzip > /dev/null 2>&1

# Install latest stable Google Chrome
!wget -q -O - https://dl.google.com/linux/linux_signing_key.pub | apt-key add -
!echo "deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main" > /etc/apt/sources.list.d/google-chrome.list
!apt-get update -qq
!apt-get install -yqq google-chrome-stable > /dev/null 2>&1

# Get Chrome version and download matching driver
import re, subprocess
chrome_version = subprocess.getoutput("google-chrome --version")
version = re.search(r"\d+\.\d+\.\d+", chrome_version).group()
print(f"Detected Chrome version: {version}")

!wget -q https://edgedl.me.gvt1.com/edgedl/chrome/chrome-for-testing/{version}/linux64/chromedriver-linux64.zip
!unzip -o chromedriver-linux64.zip
!chmod +x chromedriver-linux64/chromedriver
!mv chromedriver-linux64/chromedriver /usr/bin/chromedriver

# Install selenium and other dependencies
!pip install -q selenium beautifulsoup4 pandas pillow > /dev/null
print("Chrome + Driver installed successfully!")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
OK
W: http://dl.google.com/linux/chrome/deb/dists/stable/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Detected Chrome version: 142.0.7444
unzip:  cannot find or open chromedriver-linux64.zip, chromedriver-linux64.zip.zip or chromedriver-linux64.zip.ZIP.
chmod: cannot access 'chromedriver-linux64/chromedriver': No such file or directory
mv: cannot stat 'chromedriver-linux64/chromedriver': No such file or directory
Chrome + Driver installed successfully!


In [ ]:
import os, re, time, requests, zipfile
from urllib.parse import urljoin
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from google.colab import files
import json

# ─────────────────────── CONFIG ───────────────────────
KEYWORD = "software engineer"          # ← CHANGE THIS
OUTPUT_FOLDER = "topjobs_ads"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

def clean_name(s):
    return re.sub(r'[^\w\- ]', '_', s.strip())[:100]

# ─────────────────────── Chrome options ───────────────────────
options = Options()
options.add_argument('--headless')
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')
options.add_argument('--disable-gpu')
options.add_argument('--window-size=1920,1080')
options.add_argument('--disable-extensions')
options.add_argument('--disable-browser-side-navigation')

driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 20)

print(f"Searching for: {KEYWORD.upper()}\n")
driver.get("https://www.topjobs.lk/index.jsp")
time.sleep(3)

# Perform search
driver.find_element(By.ID, "txtKeyWord").clear()
driver.find_element(By.ID, "txtKeyWord").send_keys(KEYWORD)
driver.find_element(By.ID, "btnSearch").click()

try:
    wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "table#table")))
    print("Results page loaded")
except:
    print("No results or site changed")
    driver.quit()
    raise

time.sleep(4)
soup = BeautifulSoup(driver.page_source, "html.parser")
rows = soup.select("table#table tbody tr[onclick*='createAlert']")
print(f"Found {len(rows)} job ads\n")

jobs = []
for i, row in enumerate(rows, 1):
    onclick = row.get("onclick", "")
    m = re.search(r"createAlert\('(\d+)','([^']+)','([^']+)','([^']+)','([^']+)'\)", onclick)
    if not m: continue
    rid, ac, jc, ec, _ = m.groups()

    pos = row.find("h2").get_text(strip=True) if row.find("h2") else "N/A"
    emp = row.find("h1").get_text(strip=True) if row.find("h1") else "N/A"
    ref = row.find_all("td")[1].get_text(strip=True)

    url = f"https://www.topjobs.lk/employer/JobAdvertismentServlet?rid={rid}&ac={ac}&jc={jc}&ec={ec}&pg=applicant/vacancybyfunctionalarea.jsp"

    jobs.append({"ref": ref, "pos": pos, "emp": emp, "url": url})
    print(f"{i:2d}. [{ref}] {pos} – {emp}")

# ─────────────────────── Download ads ───────────────────────
metadata = []

for idx, job in enumerate(jobs, 1):
    print(f"\n{idx}/{len(jobs)} → {job['pos']} ({job['ref']})")
    safe = f"{job['ref']}_{clean_name(job['pos'])}"

    job_data = {
        "ref": job["ref"],
        "position": job["pos"],
        "employer": job["emp"],
        "url": job["url"],
        "type": None,
        "files": []
    }

    try:
        driver.get(job["url"])
        time.sleep(4)

        # Wait for the remark div to load
        try:
            wait.until(EC.presence_of_element_located((By.ID, "remark")))
        except:
            print(f"   ! Could not find job content div")
            continue

        soup = BeautifulSoup(driver.page_source, "html.parser")
        remark_div = soup.find("div", {"id": "remark"})

        if not remark_div:
            print(f"   ! No remark div found")
            continue

        # Check if job ad is posted as an image
        img_in_remark = remark_div.find("img")

        if img_in_remark and img_in_remark.get("src"):
            # IMAGE-BASED AD
            print(f"    Image-based ad detected")
            job_data["type"] = "image"

            src = img_in_remark.get("src")
            src = urljoin(job["url"], src)

            try:
                r = requests.get(src, timeout=12)
                if r.status_code == 200:
                    ext = src.split(".")[-1].split("?")[0][:4]
                    if not ext or len(ext) > 4:
                        ext = "png"
                    path = f"{OUTPUT_FOLDER}/{safe}_ad.{ext}"
                    open(path, "wb").write(r.content)
                    job_data["files"].append(os.path.basename(path))
                    print(f"   @ Image saved: {os.path.basename(path)}")
                else:
                    print(f"   X Failed to download image (status {r.status_code})")
            except Exception as e:
                print(f"   X Error downloading image: {e}")

        else:
            # TEXT-BASED AD
            print(f"    Text-based ad detected")
            job_data["type"] = "text"

            # Extract text content
            text_content = remark_div.get_text(separator="\n", strip=True)
            text_path = f"{OUTPUT_FOLDER}/{safe}_content.txt"
            with open(text_path, "w", encoding="utf-8") as f:
                f.write(text_content)
            job_data["files"].append(os.path.basename(text_path))
            print(f"   @ Text saved: {os.path.basename(text_path)}")

            # Take screenshot of the job ad section
            try:
                element = driver.find_element(By.ID, "remark")

                # Scroll element into view
                driver.execute_script("arguments[0].scrollIntoView(true);", element)
                time.sleep(1)

                # Get element size and position
                location = element.location
                size = element.size

                # Take full page screenshot
                screenshot_path = f"{OUTPUT_FOLDER}/{safe}_screenshot.png"
                driver.save_screenshot(screenshot_path)

                # Crop to just the job ad element using Pillow
                from PIL import Image
                img = Image.open(screenshot_path)

                # Crop with some padding
                left = max(0, location['x'] - 10)
                top = max(0, location['y'] - 10)
                right = min(img.width, location['x'] + size['width'] + 10)
                bottom = min(img.height, location['y'] + size['height'] + 10)

                cropped = img.crop((left, top, right, bottom))
                cropped.save(screenshot_path)

                job_data["files"].append(os.path.basename(screenshot_path))
                print(f"   @ Screenshot saved: {os.path.basename(screenshot_path)}")
            except Exception as e:
                print(f"   X Error taking screenshot: {e}")

        metadata.append(job_data)

    except Exception as e:
        print(f"   X Error processing job: {e}")
        continue

driver.quit()

# Save metadata
metadata_path = f"{OUTPUT_FOLDER}/metadata.json"
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)
print(f"\n Metadata saved: {metadata_path}")

print(f"SUMMARY:")
print(f"  Total jobs processed: {len(metadata)}")
print(f"  Image-based ads: {sum(1 for j in metadata if j['type'] == 'image')}")
print(f"  Text-based ads: {sum(1 for j in metadata if j['type'] == 'text')}")


# Create zip and download
print("\nCreating zip file...")
!zip -r -q topjobs_ads.zip {OUTPUT_FOLDER}
files.download("topjobs_ads.zip")
print("@ Download complete!")

Searching for: SOFTWARE ENGINEER

Results page loaded
Found 39 job ads

 1. [1434203] Customer Care Assistants | Data Entry Operators – DMS Software Engineering (Pvt) Ltd
 2. [1434198] Payroll Officer (Male) – DMS Software Engineering (Pvt) Ltd
 3. [1434195] Front Desk Receptionist (Female) – DMS Software Engineering (Pvt) Ltd
 4. [1431167] Software Engineer (Mobile Application Development) – Epic Lanka (Pvt) Ltd
 5. [1431109] Intern - Software Engineer (Mobile Application Development) – Epic Lanka (Pvt) Limited
 6. [1430463] Software Engineer - Laravel – IPOSG
 7. [1430462] Software Engineer - Full Stack – IPOSG
 8. [1430461] Senior Software Engineer - React – IPOSG
 9. [1434652] Senior Software Engineer -  JAVA – Hitachi Digital Payment Solutions Limited
10. [1434592] Full Stack Software Engineer – Ceylon Cloud Technologies (Pvt) Ltd
11. [1434558] Engineering Service Executives for Galle (Male) – Data Management Systems (Pvt) Ltd
12. [1434555] Engineering Service Executives for Matar

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ Download complete!
